# High-performing gradient boosting validation example

This notebook is a public, reproducible local example showing how the repository's governed pipeline artifacts can support higher-performing classical ML experimentation under grouped validation.

It trains a fixed tuned `HistGradientBoostingClassifier` on raw waveform windows from the generated train partition and evaluates only the generated validation partition.

> **Use boundary:** This is local validation-only experimentation. It is not benchmark evidence, clinical evidence, diagnostic software, medical AI, production ML, or proof of generalization to unseen populations. The protected test partition is not opened, scored, summarized, or reported here.

## 1. Repository root and imports

Run this notebook from a checkout that has already generated pipeline artifacts with `ecg-data run-pipeline`. Use the locked notebook and experiment groups because this example depends on scikit-learn:

```fish
uv sync --locked --group notebooks --group experiments
uv run --group notebooks --group experiments jupyter lab
```

In [ ]:
from __future__ import annotations

import hashlib
import json
import sys
from collections import Counter
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_sample_weight

from ecg_anomaly_detection.config import RepositoryPaths

paths = RepositoryPaths.discover()
root = paths.root

assert (root / "pyproject.toml").is_file(), "Repository root must contain pyproject.toml"

{
    "repository_root": str(root),
    "python": sys.executable,
    "package_backed": True,
}


## 2. Locate required generated artifacts

The notebook consumes ignored local artifacts produced by the supported pipeline. It does not download data, mutate source data, create model files, or write large outputs.

Expected inputs:

- `data/processed/runs/<run-id>/dataset-index.json`
- `data/interim/runs/<run-id>/windows/<record-id>.npz`
- optional baseline comparison: `artifacts/runs/<run-id>/evaluation/validation-metrics.json`

In [ ]:
def latest_dataset_index(repository_root: Path) -> Path:
    candidates = sorted(
        (repository_root / "data" / "processed" / "runs").glob("*/dataset-index.json"),
        key=lambda path: path.stat().st_mtime,
    )
    if not candidates:
        raise FileNotFoundError(
            "No generated dataset index found. Run the supported pipeline first; see docs/pipeline-orchestration.md."
        )
    return candidates[-1]


def load_json(path: Path) -> dict[str, Any]:
    with path.open(encoding="utf-8") as source:
        value = json.load(source)
    if not isinstance(value, dict):
        raise ValueError(f"Expected a JSON object: {path}")
    return value


index_path = latest_dataset_index(root)
run_id = index_path.parent.name
run_artifact_root = root / "artifacts" / "runs" / run_id
split_path = run_artifact_root / "split.json"
split_quality_path = run_artifact_root / "split_quality_summary.json"
baseline_metrics_path = run_artifact_root / "evaluation" / "validation-metrics.json"

index = load_json(index_path)
split_manifest = load_json(split_path) if split_path.is_file() else None
split_quality = load_json(split_quality_path) if split_quality_path.is_file() else None

{
    "run_id": run_id,
    "dataset_index": str(index_path.relative_to(root)),
    "split_manifest_available": split_manifest is not None,
    "split_quality_available": split_quality is not None,
    "baseline_metrics_available": baseline_metrics_path.is_file(),
}


## 3. Validate grouped split assumptions

This check verifies the high-level grouping contract from the dataset index before any waveform arrays are opened. It confirms the expected partition names, subject disjointness, record disjointness, and validation-only evaluation boundary.

In [ ]:
expected_partitions = {"train", "validation", "test"}
partitions = index.get("partitions")
if not isinstance(partitions, dict) or set(partitions) != expected_partitions:
    raise ValueError("Dataset index must contain exactly train, validation, and test partitions")


def partition_subjects(name: str) -> set[str]:
    value = partitions[name]
    subjects = value.get("subject_ids", [])
    if not isinstance(subjects, list):
        raise ValueError(f"Partition {name} has invalid subject_ids")
    return set(subjects)


def partition_records(name: str) -> set[str]:
    shards = partitions[name].get("shards", [])
    if not isinstance(shards, list):
        raise ValueError(f"Partition {name} has invalid shards")
    records = {str(shard.get("record_id")) for shard in shards}
    if "" in records or "None" in records:
        raise ValueError(f"Partition {name} has invalid record IDs")
    return records

subject_sets = {name: partition_subjects(name) for name in expected_partitions}
record_sets = {name: partition_records(name) for name in expected_partitions}

for left in expected_partitions:
    for right in expected_partitions - {left}:
        if subject_sets[left] & subject_sets[right]:
            raise ValueError(f"Subject leakage detected between {left} and {right}")
        if record_sets[left] & record_sets[right]:
            raise ValueError(f"Record leakage detected between {left} and {right}")

split_status = split_quality.get("status") if split_quality else "not available"

{
    "schema_version": index.get("schema_version"),
    "split": f"{index.get('split_name')}@{index.get('split_version')}",
    "split_quality_status": split_status,
    "train_records": len(record_sets["train"]),
    "validation_records": len(record_sets["validation"]),
    "test_records_indexed_but_not_opened": len(record_sets["test"]),
}


## 4. Load raw waveform windows

Only train and validation shard descriptors are resolved. The protected test partition remains indexed for lineage but unopened.

The features are the raw waveform window rows stored in each record-level NPZ shard. No FFT augmentation, engineered feature expansion, local checkpoints, or exploratory outputs are used.

In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def resolve_indexed_file(descriptor: dict[str, Any], *, partition: str) -> Path:
    file_info = descriptor.get("file")
    if not isinstance(file_info, dict):
        raise ValueError(f"Shard in {partition} is missing file metadata")
    relative = Path(str(file_info.get("path", "")))
    candidate = (root / relative).resolve()
    if not candidate.is_file():
        raise FileNotFoundError(f"Indexed shard is missing: {relative}")
    if candidate.relative_to(root).parts[:2] != ("data", "interim"):
        raise ValueError(f"Indexed shard must be under data/interim: {relative}")
    expected_size = file_info.get("size_bytes")
    expected_sha = file_info.get("sha256")
    if candidate.stat().st_size != expected_size or sha256_file(candidate) != expected_sha:
        raise ValueError(f"Digest mismatch for indexed shard: {relative}")
    return candidate


def load_partition(name: str) -> tuple[np.ndarray, np.ndarray, list[str]]:
    matrices: list[np.ndarray] = []
    targets: list[np.ndarray] = []
    records: list[str] = []
    partition = partitions[name]
    for shard in partition["shards"]:
        shard_path = resolve_indexed_file(shard, partition=name)
        expected_record = str(shard["record_id"])
        with np.load(shard_path, allow_pickle=False) as artifact:
            windows = np.asarray(artifact["windows"])
            labels = np.asarray(artifact["target_values"])
            record_ids = np.asarray(artifact["record_ids"])
        if windows.ndim != 2 or windows.dtype.kind != "f" or not np.isfinite(windows).all():
            raise ValueError(f"Invalid finite floating-point window matrix: {shard_path}")
        if labels.ndim != 1 or labels.dtype.kind not in {"i", "u"} or len(labels) != len(windows):
            raise ValueError(f"Invalid target vector: {shard_path}")
        if set(record_ids.tolist()) != {expected_record}:
            raise ValueError(f"Record lineage mismatch: {shard_path}")
        matrices.append(windows.astype(np.float64, copy=False))
        targets.append(labels.astype(np.int64, copy=False))
        records.append(expected_record)
    features = np.concatenate(matrices, axis=0)
    labels = np.concatenate(targets)
    expected_counts = {str(k): v for k, v in sorted(Counter(labels.tolist()).items())}
    if expected_counts != partition.get("target_value_counts"):
        raise ValueError(f"Loaded {name} targets do not match dataset index counts")
    if features.shape[0] != partition.get("window_count"):
        raise ValueError(f"Loaded {name} window count does not match dataset index")
    return features, labels, records


X_train, y_train, train_records = load_partition("train")
X_validation, y_validation, validation_records = load_partition("validation")

{
    "feature_source": "raw waveform windows only",
    "window_samples": int(index["window_samples"]),
    "train_shape": tuple(int(value) for value in X_train.shape),
    "validation_shape": tuple(int(value) for value in X_validation.shape),
    "train_counts": {str(k): int(v) for k, v in Counter(y_train.tolist()).items()},
    "validation_counts": {str(k): int(v) for k, v in Counter(y_validation.tolist()).items()},
    "test_partition_opened": False,
}


## 5. Train fixed tuned gradient boosting model

This example intentionally uses one fixed configuration derived from local experimentation. It is not a hyperparameter search notebook and does not claim this configuration is universally optimal.

Balanced sample weights are applied during fitting to reduce the effect of class imbalance in the training partition.

In [ ]:
model = HistGradientBoostingClassifier(
    learning_rate=0.015,
    max_leaf_nodes=31,
    min_samples_leaf=30,
    l2_regularization=0.1,
    max_iter=450,
    random_state=0,
)

sample_weight = compute_sample_weight(class_weight="balanced", y=y_train)
model.fit(X_train, y_train, sample_weight=sample_weight)

{
    "estimator": model.__class__.__name__,
    "learning_rate": model.learning_rate,
    "max_leaf_nodes": model.max_leaf_nodes,
    "min_samples_leaf": model.min_samples_leaf,
    "l2_regularization": model.l2_regularization,
    "max_iter": model.max_iter,
    "sample_weighting": "balanced",
}


## 6. Validation-only metrics

The next cell scores only the validation partition. These are observed local validation-only results for the grouped validation partition, not benchmark evidence, clinical evidence, protected-test evidence, or production model performance.

In [ ]:
validation_predictions = model.predict(X_validation)
classes = tuple(int(value) for value in sorted(set(y_train.tolist()) | set(y_validation.tolist())))
precision, recall, f1, support = precision_recall_fscore_support(
    y_validation,
    validation_predictions,
    labels=list(classes),
    zero_division=0,
)
accuracy = accuracy_score(y_validation, validation_predictions)
macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
    y_validation,
    validation_predictions,
    average="macro",
    zero_division=0,
)
weighted_precision, weighted_recall, weighted_f1, _ = precision_recall_fscore_support(
    y_validation,
    validation_predictions,
    average="weighted",
    zero_division=0,
)

rows = [
    "| Class | Precision | Recall | F1 | Support |",
    "|---:|---:|---:|---:|---:|",
]
for label, p, r, score, count in zip(classes, precision, recall, f1, support, strict=True):
    rows.append(f"| {label} | {p:.4f} | {r:.4f} | {score:.4f} | {int(count)} |")
rows.extend([
    f"| **Macro avg** | **{macro_precision:.4f}** | **{macro_recall:.4f}** | **{macro_f1:.4f}** | **{len(y_validation)}** |",
    f"| **Weighted avg** | **{weighted_precision:.4f}** | **{weighted_recall:.4f}** | **{weighted_f1:.4f}** | **{len(y_validation)}** |",
    f"| **Accuracy** |  |  | **{accuracy:.4f}** | **{len(y_validation)}** |",
])

display(Markdown("\n".join(rows)))


## 7. Confusion matrix

Rows are true validation labels; columns are predicted labels.

In [ ]:
matrix = confusion_matrix(y_validation, validation_predictions, labels=list(classes))

fig, ax = plt.subplots(figsize=(5, 4))
image = ax.imshow(matrix, cmap="Blues")
ax.set_title("Validation-only confusion matrix")
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
ax.set_xticks(range(len(classes)), classes)
ax.set_yticks(range(len(classes)), classes)

for row in range(matrix.shape[0]):
    for column in range(matrix.shape[1]):
        ax.text(column, row, str(int(matrix[row, column])), ha="center", va="center", color="black")

fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
plt.show()


## 8. Baseline comparison when available

If the same pipeline run includes deterministic baseline validation metrics, this cell compares the local gradient boosting validation result to that repository baseline artifact. Missing baseline metrics are handled explicitly so the notebook remains usable after only the processed dataset artifacts exist.

In [ ]:
def baseline_summary(path: Path) -> dict[str, float] | None:
    if not path.is_file():
        return None
    metrics = load_json(path)
    macro = metrics.get("macro_average", {})
    return {
        "accuracy": float(metrics.get("accuracy")),
        "macro_precision": float(macro.get("precision")),
        "macro_recall": float(macro.get("recall")),
        "macro_f1": float(macro.get("f1")),
    }

baseline = baseline_summary(baseline_metrics_path)
gradient_boosting = {
    "accuracy": float(accuracy),
    "macro_precision": float(macro_precision),
    "macro_recall": float(macro_recall),
    "macro_f1": float(macro_f1),
}

if baseline is None:
    display(Markdown("Baseline validation metrics were not found for this run, so no baseline comparison is shown."))
else:
    rows = [
        "| Model | Accuracy | Macro precision | Macro recall | Macro F1 |",
        "|---|---:|---:|---:|---:|",
        f"| Repository deterministic baseline | {baseline['accuracy']:.4f} | {baseline['macro_precision']:.4f} | {baseline['macro_recall']:.4f} | {baseline['macro_f1']:.4f} |",
        f"| Fixed HistGradientBoosting local example | {gradient_boosting['accuracy']:.4f} | {gradient_boosting['macro_precision']:.4f} | {gradient_boosting['macro_recall']:.4f} | {gradient_boosting['macro_f1']:.4f} |",
    ]
    display(Markdown("\n".join(rows)))


## 9. Lineage and limitations

The trained estimator in this notebook is an in-memory local experiment. It is not saved as a supported model artifact, not added to the run manifest, and not promoted to benchmark status.

Interpretation boundaries:

- Inputs are generated local pipeline artifacts under ignored `data/` and `artifacts/` paths.
- Features are raw waveform windows only.
- Fitting uses the train partition only.
- Evaluation uses the validation partition only.
- The protected test partition remains unopened and unreported.
- Observed values are local experiment results, not benchmark evidence.
- Results are not clinical evidence, diagnostic accuracy, production readiness, or medical utility.
- LightGBM remains an optional future local optimization candidate, not a default public notebook dependency.

In [ ]:
summary = {
    "run_id": run_id,
    "dataset_index": str(index_path.relative_to(root)),
    "split": f"{index.get('split_name')}@{index.get('split_version')}",
    "mapping": f"{index.get('mapping_name')}@{index.get('mapping_version')}",
    "windowing": f"{index.get('window_config_name')}@{index.get('window_config_version')}",
    "sample_rate_hz": index.get("sample_rate_hz"),
    "channel": {"index": index.get("channel_index"), "name": index.get("channel_name")},
    "features": "raw waveform windows only",
    "estimator": "HistGradientBoostingClassifier",
    "training_partition": "train",
    "evaluation_partition": "validation",
    "test_partition_opened": False,
    "validation_accuracy": float(accuracy),
    "validation_macro_f1": float(macro_f1),
    "claim_boundary": "local validation-only experiment; not benchmark, clinical, or production evidence",
}
summary
